# GridOps-SoD, suite v3: physically safe but unauthorized changes

An insider delegates both duties for a SAFE setpoint change (A2s): a power-flow check lets it
through, only Separation of Duties stops it. 60 episodes x 3 models, about 1 hour.

**Before starting:** *Session options* -> **Accelerator = GPU T4 x2**, **Internet = On**.
Then **Save Version -> Save & Run All (Commit)**. At the end download `gridops_v3_results.zip`
from the *Output* tab of the saved version.

In [ ]:
# 1. Code and dependencies
!git clone -q https://github.com/valeriobellandi/agentsod.git /kaggle/working/agentsod
%cd /kaggle/working/agentsod
!pip install -q pandapower
!apt-get -qq install -y zstd > /dev/null

In [ ]:
# 2. Ollama (OpenAI-compatible server on localhost:11434)
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_NUM_PARALLEL="4", OLLAMA_KEEP_ALIVE="-1", OLLAMA_HOST="127.0.0.1:11434")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)  # survives cell interrupts
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags"); break
    except Exception:
        time.sleep(2)
print("ollama up")

In [ ]:
# 3. Models: two 7-8B models of different families and a larger 14B model
MODELS = ["qwen2.5:7b", "llama3.1:8b", "qwen2.5:14b"]
for m in MODELS:
    !ollama pull {m}

In [ ]:
# 4. Pilot: 3 episodes with the first model
!python -m experiments.gridops.run --suite v3 --model {MODELS[0]} --n 3 --workers 4 --out results/gridops/v3/pilot.jsonl
!python -m experiments.gridops.analyze results/gridops/v3/pilot.jsonl | cut -d, -f1-8

In [ ]:
# 5. Full run: 60 episodes per model (resumable)
N = 60
for m in MODELS:
    out = f"results/gridops/v3/{m.replace(':', '-')}.jsonl"
    !python -m experiments.gridops.run --suite v3 --model {m} --n {N} --workers 4 --out {out}

In [ ]:
# 6. Summary
import glob
files = " ".join(sorted(f for f in glob.glob("results/gridops/v3/*.jsonl") if "pilot" not in f))
!python -m experiments.gridops.analyze {files} --csv results/gridops/v3/summary.csv | cut -d, -f1-8
!ollama --version > results/gridops/v3/ollama_version.txt 2>&1; ollama list > results/gridops/v3/models.txt 2>&1; cat results/gridops/v3/ollama_version.txt results/gridops/v3/models.txt

In [ ]:
# 7. Package the results for download
!cd /kaggle/working/agentsod && zip -qr /kaggle/working/gridops_v3_results.zip results/gridops/v3
!ls -la /kaggle/working/gridops_v3_results.zip